# MatchFormer Epipolar Fine-tuning — Google Colab

Fine-tunes MatchFormer Lite-LA with epipolar supervision on ScanNet.
Checkpoints are saved to **Google Drive** so training survives Colab disconnects.

**Run order (fresh session):** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7  
**Resume after disconnect:** Cell 1 → 3 → 4 → 7 (skip 2, 5, 6)

## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ZIPS    = '/content/drive/MyDrive/final_proj/scannet_zips'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/final_proj/cr7'

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Zips dir:       {DRIVE_ZIPS}')
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

Mounted at /content/drive
Zips dir:       /content/drive/MyDrive/final_proj/scannet_zips
Checkpoint dir: /content/drive/MyDrive/final_proj/cr7


## Cell 2 — Clone Repo & Install Dependencies
*(Skip on resume — repo and packages persist for the session)*

In [2]:
REPO_URL = 'https://github.com/sid-raj-uc/match-former.git'
REPO_DIR = '/content/final_proj'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    !git clone -b develop {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !cd {REPO_DIR} && git fetch origin && git checkout -f develop && git pull origin develop

%cd /content/final_proj/MatchFormer



Cloning repo...
Cloning into '/content/final_proj'...
remote: Enumerating objects: 311, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 311 (delta 72), reused 72 (delta 33), pack-reused 192 (from 1)
Receiving objects: 100% (311/311), 133.18 MiB | 16.59 MiB/s, done.
Resolving deltas: 100% (155/155), done.
/content/final_proj/MatchFormer


In [3]:
!pip install -q pytorch-lightning einops kornia timm loguru yacs
!pip install -q opencv-python-headless
print('Done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 75.9 MB/s eta 0:00:00
Done.


## Cell 3 — Verify Zips on Drive

In [4]:
import glob, os

%cd /content/final_proj/MatchFormer

zips = sorted(glob.glob(os.path.join(DRIVE_ZIPS, '*.zip')))
print(f'Found {len(zips)} zip(s) on Drive:')
for z in zips:
    size_gb = os.path.getsize(z) / 1e9
    print(f'  {os.path.basename(z)}  ({size_gb:.2f} GB)')

/content/final_proj/MatchFormer
Found 11 zip(s) on Drive:
  scene0000_00.zip  (2.24 GB)
  scene0001_00.zip  (0.58 GB)
  scene0002_00.zip  (2.22 GB)
  scene0003_00.zip  (0.60 GB)
  scene0004_00.zip  (0.28 GB)
  scene0005_00.zip  (0.36 GB)
  scene0006_00.zip  (0.77 GB)
  scene0007_00.zip  (0.40 GB)
  scene0008_00.zip  (0.39 GB)
  scene0009_00.zip  (0.48 GB)
  scene0010_00.zip  (0.81 GB)


## Cell 4 — Extract Zips to Local SSD

Copies zips from Drive to Colab's local SSD and extracts them.
Drive has high per-file latency — local SSD is much faster for the dataloader.
Takes ~5–10 min once per session; already-extracted scenes are skipped.

In [5]:
import os, glob, shutil, subprocess, time

LOCAL_DATA = '/content/scannet_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

zips = sorted(glob.glob(os.path.join(DRIVE_ZIPS, '*.zip')))
print(f'Found {len(zips)} zip(s) to extract\n')

failed = []

for i, zip_drive in enumerate(zips, 1):
    scene = os.path.basename(zip_drive).replace('.zip', '')
    dst = os.path.join(LOCAL_DATA, scene)
    prefix = f'[{i}/{len(zips)}] {scene}'

    if os.path.isdir(dst) and len(glob.glob(os.path.join(dst, 'exported/color/*.jpg'))) > 0:
        n = len(glob.glob(os.path.join(dst, 'exported/color/*.jpg')))
        print(f'{prefix} — already extracted ({n} frames), skipping')
        continue

    size_gb = os.path.getsize(zip_drive) / 1e9
    zip_local = f'/content/{scene}.zip'

    t0 = time.time()
    print(f'{prefix} — copying {size_gb:.2f} GB from Drive...', end=' ', flush=True)
    shutil.copy2(zip_drive, zip_local)
    print(f'done ({time.time()-t0:.0f}s). Extracting...', end=' ', flush=True)

    t1 = time.time()
    result = subprocess.run(['unzip', '-q', zip_local, '-d', LOCAL_DATA])
    os.remove(zip_local)

    if result.returncode != 0:
        print(f'FAILED (unzip error {result.returncode}) — zip may be corrupted on Drive.')
        failed.append(scene)
    else:
        print(f'done ({time.time()-t1:.0f}s).')

total = len(glob.glob(os.path.join(LOCAL_DATA, '*/exported/color/*.jpg')))
print(f'\nAll done. {total} frames in {LOCAL_DATA}')
if failed:
    print(f'\nFailed scenes (re-upload zips to Drive): {failed}')

Found 11 zip(s) to extract

[1/11] scene0000_00 — copying 2.24 GB from Drive... done (58s). Extracting... done (13s).
[2/11] scene0001_00 — copying 0.58 GB from Drive... done (14s). Extracting... done (3s).
[3/11] scene0002_00 — copying 2.22 GB from Drive... done (64s). Extracting... done (13s).
[4/11] scene0003_00 — copying 0.60 GB from Drive... done (21s). Extracting... done (4s).
[5/11] scene0004_00 — copying 0.28 GB from Drive... done (8s). Extracting... done (2s).
[6/11] scene0005_00 — copying 0.36 GB from Drive... done (10s). Extracting... done (2s).
[7/11] scene0006_00 — copying 0.77 GB from Drive... done (20s). Extracting... done (5s).
[8/11] scene0007_00 — copying 0.40 GB from Drive... done (10s). Extracting... done (2s).
[9/11] scene0008_00 — copying 0.39 GB from Drive... done (11s). Extracting... done (2s).
[10/11] scene0009_00 — copying 0.48 GB from Drive... done (16s). Extracting... done (3s).
[11/11] scene0010_00 — copying 0.81 GB from Drive... done (21s). Extracting... d

## Cell 5 — Download Pretrained Weights
*(Skip on resume — weights persist for the session)*

In [6]:
WEIGHTS_PATH  = 'model/weights/indoor-lite-LA.ckpt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/final_proj/MatchFormer/model/weights/indoor-lite-LA.ckpt'

os.makedirs('model/weights', exist_ok=True)

if os.path.exists(WEIGHTS_PATH):
    print('Weights already present')
elif os.path.exists(DRIVE_WEIGHTS):
    shutil.copy(DRIVE_WEIGHTS, WEIGHTS_PATH)
    print('Copied weights from Drive')
else:
    print('WARNING: weights not found. Upload indoor-lite-LA.ckpt to Drive at:')
    print(' ', DRIVE_WEIGHTS)

Copied weights from Drive


## Cell 6 — Verify GPU & Sanity Check

Runs 50 steps on 5 pairs to confirm the pipeline works end-to-end.  
Loss should decrease. Takes ~1 min on L4.

In [7]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — no GPU!"}')
print(f'CUDA: {torch.cuda.is_available()}')

# # Sanity check: 5 pairs, 50 steps, reads from Drive (fine for 5 pairs)
# !python train_finetune.py \
#     --overfit \
#     --data_dir {DATA_ROOT} \
#     --checkpoint_dir {DRIVE_CKPT_DIR}/overfit \
#     --batch 4 \
#     --steps 50

# print('Sanity check done — loss should be decreasing!')

GPU: NVIDIA A100-SXM4-40GB
CUDA: True


## Cell 7 — Full Fine-Tuning

**Auto-resume is on** — if Colab disconnects, re-run Cell 1 → 3 → 4 → this cell.  
It will pick up from the last checkpoint in `DRIVE_CKPT_DIR` automatically.

Key settings:
- `--data_dir` reads from **local SSD** (fast)
- `--precision bf16` — ~2x speedup on L4 via native bf16 tensor cores
- `--tau 50.0` — soft epipolar mask, better for multi-scene diversity
- `--save_every 500` — saves to Drive every 500 steps (20 checkpoints total)

In [8]:
import wandb
wandb.login()

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python train_finetune.py \
    --data_dir {LOCAL_DATA} \
    --checkpoint_dir {DRIVE_CKPT_DIR} \
    --steps 5000 \
    --tau 50.0 \
    --batch 4 \
    --workers 4 \
    --lr 1e-5 \
    --neg_per_pos 15 \
    --override_lr \
    --save_every 500 \
    --precision bf16 \
    --wandb \
    --wandb_project matchformer-finetune \
    --wandb_run phase2-lr1e5-neg15-alpha05-gamma0-b8

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sidraj (sidraj-university-of-chicago-charter-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[Auto-resume] No checkpoint found — starting from scratch.
  Scenes found: 11
    scene0000_00: 5558 pairs
    scene0001_00: 1524 pairs
    scene0002_00: 5173 pairs
    scene0003_00: 1716 pairs
    scene0004_00: 909 pairs
    scene0005_00: 1139 pairs
    scene0006_00: 2140 pairs
    scene0007_00: 1021 pairs
    scene0008_00: 1018 pairs
    scene0009_00: 960 pairs
    scene0010_00: 2493 pairs
  Total pairs: 23651
Dataset: 23651 pairs
2026-03-19 19:40:19.281 | INFO     | model.lightning_loftr:__init__:54 - Load 'model/weights/indoor-lite-LA.ckpt' as pretrained checkpoint
[Phase 2] LR override active: will set lr=1e-05 after checkpoint load
[W&B] Logging to project='matchformer-finetune' run='phase2-lr1e5-neg15-alpha05

## Cell 8 — Benchmark After Training

In [ ]:
import re, glob, shutil, os

LOCAL_DATA = '/content/scannet_data'
LOCAL_CKPT = 'model/weights/epipolar-finetuned.ckpt'

# Pick the highest epipolar-step=N.ckpt as the final weights
step_ckpts = [p for p in glob.glob(f'{DRIVE_CKPT_DIR}/epipolar-*.ckpt')
              if re.search(r'epipolar-step=(\d+)\.ckpt', p)]
if step_ckpts:
    TRAINED_CKPT = max(step_ckpts, key=lambda p: int(re.search(r'epipolar-step=(\d+)', p).group(1)))
else:
    TRAINED_CKPT = f'{DRIVE_CKPT_DIR}/last.ckpt'

print(f'Using checkpoint: {TRAINED_CKPT}')
os.makedirs('model/weights', exist_ok=True)
shutil.copy(TRAINED_CKPT, LOCAL_CKPT)
print(f'Copied to {LOCAL_CKPT}')

!python run_benchmark.py \
    --ckpt {LOCAL_CKPT} \
    --data_dir {LOCAL_DATA}/scene0000_00/exported \
    --num_pairs 100 \
    2>&1 | tee benchmark_epipolar.txt

shutil.copy('benchmark_epipolar.txt', f'{DRIVE_CKPT_DIR}/benchmark_epipolar.txt')
print('Results saved to Drive.')